# EDA — SCHWIMMBADOGD.csv (Schwimmbäder / public pools)

City of Vienna open data export. Source file: `data/raw/SCHWIMMBADOGD.csv`. Run all cells to
reproduce the findings below.

In [1]:
import pandas as pd
import re

df = pd.read_csv("../data/raw/SCHWIMMBADOGD.csv")
df.shape

(46, 26)

## Columns & dtypes

In [2]:
df.dtypes

FID                                 str
SHAPE                               str
NAME                                str
ADRESSE                             str
WEBLINK1                            str
WEBLINK2                            str
BEZIRK                            int64
AUSLASTUNG_TAG_0                    str
AUSLASTUNG_AMPEL_KAT_TXT_0          str
AUSLASTUNG_AMPEL_KATEGORIE_0      int64
AUSLASTUNG_TAG_1                    str
AUSLASTUNG_AMPEL_KAT_TXT_1          str
AUSLASTUNG_AMPEL_KATEGORIE_1    float64
AUSLASTUNG_TAG_2                    str
AUSLASTUNG_AMPEL_KAT_TXT_2          str
AUSLASTUNG_AMPEL_KATEGORIE_2    float64
AUSLASTUNG_TAG_3                    str
AUSLASTUNG_AMPEL_KAT_TXT_3          str
AUSLASTUNG_AMPEL_KATEGORIE_3    float64
TIMESTAMP_MODIFIED_FORMAT           str
SE_SDO_ROWID                      int64
SE_ANNO_CAD_DATA                float64
AUSLASTUNG_TAG_0_TEXT               str
AUSLASTUNG_TAG_1_TEXT               str
AUSLASTUNG_TAG_2_TEXT               str


## Missing values

In [3]:
df.isnull().sum()

FID                              0
SHAPE                            0
NAME                             0
ADRESSE                          0
WEBLINK1                         1
WEBLINK2                        24
BEZIRK                           0
AUSLASTUNG_TAG_0                13
AUSLASTUNG_AMPEL_KAT_TXT_0      13
AUSLASTUNG_AMPEL_KATEGORIE_0     0
AUSLASTUNG_TAG_1                13
AUSLASTUNG_AMPEL_KAT_TXT_1      13
AUSLASTUNG_AMPEL_KATEGORIE_1    13
AUSLASTUNG_TAG_2                13
AUSLASTUNG_AMPEL_KAT_TXT_2      13
AUSLASTUNG_AMPEL_KATEGORIE_2    13
AUSLASTUNG_TAG_3                13
AUSLASTUNG_AMPEL_KAT_TXT_3      13
AUSLASTUNG_AMPEL_KATEGORIE_3    13
TIMESTAMP_MODIFIED_FORMAT       13
SE_SDO_ROWID                     0
SE_ANNO_CAD_DATA                46
AUSLASTUNG_TAG_0_TEXT           13
AUSLASTUNG_TAG_1_TEXT           13
AUSLASTUNG_TAG_2_TEXT           13
AUSLASTUNG_TAG_3_TEXT           13
dtype: int64

## Duplicate check

In [4]:
print("Duplicate full rows:", df.duplicated().sum())
print("Duplicate NAME:", df["NAME"].duplicated().sum())

Duplicate full rows: 0
Duplicate NAME: 0


## Parse geometry

`SHAPE` is a WKT geometry string. Parse into `lon`/`lat` (works for the first
coordinate pair even if the geometry is a line/polygon) and sanity-check the
range against Vienna's bounding box.

In [5]:
def parse_first_point(s):
    m = re.search(r"(-?\d+\.\d+)\s+(-?\d+\.\d+)", str(s))
    if m:
        return float(m.group(1)), float(m.group(2))
    return None, None

df["lon"], df["lat"] = zip(*df["SHAPE"].map(parse_first_point))
print("Unparseable SHAPE values:", df["lon"].isnull().sum())
print("lon range:", df["lon"].min(), "-", df["lon"].max())
print("lat range:", df["lat"].min(), "-", df["lat"].max())

Unparseable SHAPE values: 0
lon range: 16.2232736716689 - 16.458293698542576
lat range: 48.132669208719406 - 48.30011078517033


## District (`BEZIRK`) distribution

In [6]:
df["BEZIRK"].value_counts(dropna=False).sort_index()

BEZIRK
1     1
2     2
3     2
5     1
7     1
10    4
11    2
12    1
13    2
14    4
15    1
16    4
17    2
18    2
19    3
20    1
21    5
22    6
23    2
Name: count, dtype: int64

## Live occupancy data (`AUSLASTUNG_*`)

This "static" export bundles daily occupancy/utilization traffic-light indicators
for the next few days — genuinely live-ish data sitting inside an open-data CSV.
Check which pools don't have this data (missing, not necessarily broken) and what
the traffic-light categories look like.

In [7]:
auslastung_cols = [c for c in df.columns if c.startswith("AUSLASTUNG")]
print("AUSLASTUNG-related columns:", auslastung_cols)

print("\nPools with NO occupancy data in this export:")
print(df.loc[df["AUSLASTUNG_TAG_0"].isnull(), "NAME"].tolist())

print("\nTraffic-light category values (day 0):")
print(df["AUSLASTUNG_AMPEL_KAT_TXT_0"].value_counts(dropna=False))

AUSLASTUNG-related columns: ['AUSLASTUNG_TAG_0', 'AUSLASTUNG_AMPEL_KAT_TXT_0', 'AUSLASTUNG_AMPEL_KATEGORIE_0', 'AUSLASTUNG_TAG_1', 'AUSLASTUNG_AMPEL_KAT_TXT_1', 'AUSLASTUNG_AMPEL_KATEGORIE_1', 'AUSLASTUNG_TAG_2', 'AUSLASTUNG_AMPEL_KAT_TXT_2', 'AUSLASTUNG_AMPEL_KATEGORIE_2', 'AUSLASTUNG_TAG_3', 'AUSLASTUNG_AMPEL_KAT_TXT_3', 'AUSLASTUNG_AMPEL_KATEGORIE_3', 'AUSLASTUNG_TAG_0_TEXT', 'AUSLASTUNG_TAG_1_TEXT', 'AUSLASTUNG_TAG_2_TEXT', 'AUSLASTUNG_TAG_3_TEXT']

Pools with NO occupancy data in this export:
['Schönbrunner Bad', 'Hermannbad', 'Penzinger Bad', 'Therme Wien', 'Straßenbahnerbad', 'Neuwaldegger Bad', 'Bundesbad Alte Donau', 'Brausebad Friedrich-Kaiser-Gasse', 'Apostelbad', 'Stadionbad', 'Stadthallenbad', 'Badeschiff Wien', 'Strandbad Stadlau']

Traffic-light category values (day 0):
AUSLASTUNG_AMPEL_KAT_TXT_0
Noch Platz     31
NaN            13
geschlossen     2
Name: count, dtype: int64


## Sample rows

In [8]:
df[["NAME", "BEZIRK", "ADRESSE", "AUSLASTUNG_AMPEL_KAT_TXT_0", "lon", "lat"]].sample(5, random_state=1)

,NAME,BEZIRK,ADRESSE,AUSLASTUNG_AMPEL_KAT_TXT_0,lon,lat
34,Strandbad Angelibad,21,"21., An der oberen Alten Donau",Noch Platz,16.403348,48.248770
42,Badeschiff Wien,1,"1., An der Donaukanallände, zwischen Schwedenb...",NaN,16.382021,48.211802
2,Familienbad Hofferplatz,16,"16., Hofferplatz 11",Noch Platz,16.333372,48.209088
21,Höpflerbad,23,"23., Endresstraße 24-26",Noch Platz,16.285886,48.147531
3,Familienbad-Hugo-Wolf-Park,19,"19., Hugo-Wolf-Park, Eingang Dänenstraße",Noch Platz,16.334692,48.239820


## Findings

**Basics:** 46 records, 26 columns — richest schema of the six thanks to the
occupancy fields.

**Missing values:** 13 of 46 pools have no `AUSLASTUNG_*` data at all (see the
named list above — includes some well-known outdoor pools, so this isn't purely
an "indoor pools don't get monitored" pattern; the actual reason isn't obvious
from the data alone and would need checking against the source website).

**No duplicates.**

**Live data:** confirmed — `AUSLASTUNG_AMPEL_KAT_TXT_0..3` give a plain-text
occupancy status (e.g. "Noch Platz" = still room, "geschlossen" = closed) for
today plus the next 3 days, refreshed on each export. This is the strongest
live/temporal signal found outside the Wiener Linien data so far, and fits
directly into the one-pager's "(live) mobility conditions" framing if pools are
included as an activity-planning destination.

**Suitability for the KG:** good candidate, with a bonus: it's the second
dataset (after Badestellen's water-quality dates) carrying genuinely temporal
data. Suggested mapping: `NAME` → `poi:name`, `BEZIRK` → `poi:district`,
`ADRESSE` → `poi:address`, `lon`/`lat` → `geo:long`/`geo:lat`,
`AUSLASTUNG_AMPEL_KAT_TXT_0..3` → a time-indexed `poi:occupancyStatus` property
if the live-reasoning angle gets pursued, otherwise safe to drop for a first
static pass.